# Notebook 06: Benchmark Evaluation on Held-Out Test Signers
Evaluates checkpoint against held-out test split, computing SacreBLEU (1..4), chrF++, WER, and semantic accuracy.


In [ ]:
import os, sys, json, torch
sys.path.insert(0, '.')
from ml.inference.predictor import ISLPredictor
from ml.training.metrics import compute_bleu, compute_chrf, compute_wer
from ml.data.dataset import ISLDataset

predictor = ISLPredictor()
test_manifest = 'data/manifests/test_manifest.json'

with open(test_manifest, 'r', encoding='utf-8') as f:
    test_samples = json.load(f)

print(f'Evaluating {len(test_samples)} test samples...')
hypotheses = []
references = []

for sample in test_samples:
    if sample.get('pose_path') and os.path.exists(sample['pose_path']):
        import numpy as np
        features = np.load(sample['pose_path'])
        res = predictor.predict_pose_sequence(features)
        hypotheses.append(res['english_text'])
    else:
        hypotheses.append('unknown')
    references.append(sample['english_text'])

bleu = compute_bleu(hypotheses, references)
chrf = compute_chrf(hypotheses, references)
wer = compute_wer(hypotheses, references)

print('=== TEST SET BENCHMARK RESULTS ===')
print(f'BLEU-1: {bleu.get("bleu_1")}')
print(f'BLEU-2: {bleu.get("bleu_2")}')
print(f'BLEU-3: {bleu.get("bleu_3")}')
print(f'BLEU-4: {bleu.get("bleu_4")}')
print(f'chrF++: {chrf}')
print(f'WER:    {wer}%')
